# Exercise 03: Database Interfacing with SQL
# Cluster 5 - Image, Audio, Text and Database Processing
# Framework: EU AI Act Article 17 (Quality Management), DPDP Act India 2023
# Case Study: NCII 🔴

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.45.1


In [2]:
# Create an in-memory database for NCII incident tracking
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create reports table
cursor.execute('''
    CREATE TABLE reports (
        report_id INTEGER PRIMARY KEY,
        reported_at TEXT,
        platform TEXT,
        content_type TEXT,
        victim_notified INTEGER,
        content_removed INTEGER,
        account_deleted INTEGER,
        evidence_preserved INTEGER
    )
''')

print("Database created successfully")

Database created successfully


In [3]:
# Insert synthetic NCII incident reports
incidents = [
    ('2024-01-15 10:23:00', 'Platform A', 'deepfake_image', 1, 1, 0, 1),
    ('2024-01-16 14:45:00', 'Platform B', 'deepfake_video', 0, 0, 1, 0),
    ('2024-01-17 09:12:00', 'Platform A', 'manipulated_image', 1, 1, 0, 1),
    ('2024-01-18 16:30:00', 'Platform C', 'deepfake_image', 0, 1, 1, 0),
    ('2024-01-19 11:00:00', 'Platform B', 'deepfake_video', 1, 0, 0, 1),
    ('2024-01-20 08:45:00', 'Platform A', 'manipulated_image', 0, 0, 1, 0),
    ('2024-01-21 13:20:00', 'Platform C', 'deepfake_image', 1, 1, 0, 1),
    ('2024-01-22 15:10:00', 'Platform B', 'deepfake_video', 0, 0, 1, 0),
]

cursor.executemany('''
    INSERT INTO reports 
    (reported_at, platform, content_type, victim_notified, 
     content_removed, account_deleted, evidence_preserved)
    VALUES (?, ?, ?, ?, ?, ?, ?)
''', incidents)

conn.commit()
print(f"Inserted {len(incidents)} incident reports")

Inserted 8 incident reports


In [4]:
# Query: show the relationship between account deletion and evidence preservation
query = '''
    SELECT report_id, reported_at, platform, account_deleted, evidence_preserved
    FROM reports
    ORDER BY report_id
'''

results = cursor.execute(query).fetchall()
df = pd.DataFrame(results, columns=['report_id', 'reported_at', 'platform', 
                                     'account_deleted', 'evidence_preserved'])
print(df)
print("\n--- Governance Pattern ---")
print(df.groupby('account_deleted')['evidence_preserved'].mean())

   report_id          reported_at    platform  account_deleted  \
0          1  2024-01-15 10:23:00  Platform A                0   
1          2  2024-01-16 14:45:00  Platform B                1   
2          3  2024-01-17 09:12:00  Platform A                0   
3          4  2024-01-18 16:30:00  Platform C                1   
4          5  2024-01-19 11:00:00  Platform B                0   
5          6  2024-01-20 08:45:00  Platform A                1   
6          7  2024-01-21 13:20:00  Platform C                0   
7          8  2024-01-22 15:10:00  Platform B                1   

   evidence_preserved  
0                   1  
1                   0  
2                   1  
3                   0  
4                   1  
5                   0  
6                   1  
7                   0  

--- Governance Pattern ---
account_deleted
0    1.0
1    0.0
Name: evidence_preserved, dtype: float64


In [5]:
# Which platforms are worst at preserving evidence?
query2 = '''
    SELECT platform,
           COUNT(*) as total_incidents,
           SUM(victim_notified) as victims_notified,
           SUM(content_removed) as content_removed,
           SUM(evidence_preserved) as evidence_preserved
    FROM reports
    GROUP BY platform
'''

results2 = cursor.execute(query2).fetchall()
df2 = pd.DataFrame(results2, columns=['platform', 'total_incidents', 
                                       'victims_notified', 'content_removed', 
                                       'evidence_preserved'])
print(df2)

     platform  total_incidents  victims_notified  content_removed  \
0  Platform A                3                 2                2   
1  Platform B                3                 1                0   
2  Platform C                2                 1                2   

   evidence_preserved  
0                   2  
1                   1  
2                   1  


## Governance Reflection: Database Evidence Preservation

This database reveals a critical governance failure: platforms are deleting evidence when perpetrators delete accounts. The data shows a perfect inverse relationship — when account_deleted = 1, evidence_preserved = 0 always. This means victims lose the ability to prove what happened to them.

Platform B demonstrates the worst practice: 0% content removal rate and only 33% evidence preservation. Platform C is better but still incomplete.

A regulator auditing this data would require:

1. **Mandatory evidence preservation:** Evidence must be preserved in a separate audit log, independent of account deletion. EU AI Act Article 17 requires quality management systems — this means documented, timestamped logs that cannot be deleted.

2. **Higher content removal rates:** Especially for deepfake and manipulated imagery, removal must be near-immediate. Platform B's 0% removal rate is unacceptable.

3. **Victim notification:** Every victim must be notified that NCII was detected and removed, regardless of whether the perpetrator's account was deleted. This is required by EU AI Act Article 20 and DPDP Act Section 8.

Without these requirements, platforms can delete accountability itself.